<a href="https://colab.research.google.com/github/Omtambat/Wine-Dataset-Mini-Project-using-Scikit-learn/blob/main/Wine_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report
)


wine = load_wine()

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

df["target"] = wine.target

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

print("\nWine Classes:")
print(wine.target_names)

# 178 samples, 13 chemical features, 3 wine classes

# TASK 1: REGRESSION
# Predict alcohol using the remaining 12 features

y = df["alcohol"]


X = df.drop(columns=["alcohol", "target"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train Linear Regression
lr = LinearRegression()

lr.fit(X_train, y_train)

# Predictions
y_pred = lr.predict(X_test)

# Evaluation
print("R2 Score:", round(r2_score(y_test, y_pred), 4))

print(
    "Mean Squared Error:",
    round(mean_squared_error(y_test, y_pred), 4)
)


scaler_reg = StandardScaler()

X_train_scaled = scaler_reg.fit_transform(X_train)

lr_scaled = LinearRegression()

lr_scaled.fit(X_train_scaled, y_train)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_scaled.coef_,
    "Absolute Importance": np.abs(lr_scaled.coef_)
})

importance = importance.sort_values(
    by="Absolute Importance",
    ascending=False
)

print("\nTop 2 Features Influencing Alcohol:")
print(importance.head(2))


# TASK 2: CLASSIFICATION
# Predict wine cultivar: target = 0, 1, or 2

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression BEFORE scaling

log_before = LogisticRegression(max_iter=1000)

log_before.fit(X_train, y_train)

pred_log_before = log_before.predict(X_test)

acc_log_before = accuracy_score(
    y_test, pred_log_before
)

# Logistic Regression AFTER scaling

log_after = LogisticRegression(max_iter=1000)

log_after.fit(X_train_scaled, y_train)

pred_log_after = log_after.predict(X_test_scaled)

acc_log_after = accuracy_score(
    y_test, pred_log_after
)

# KNN BEFORE scaling

knn_before = KNeighborsClassifier(n_neighbors=5)

knn_before.fit(X_train, y_train)

pred_knn_before = knn_before.predict(X_test)

acc_knn_before = accuracy_score(
    y_test, pred_knn_before
)

# KNN AFTER scaling

knn_after = KNeighborsClassifier(n_neighbors=5)

knn_after.fit(X_train_scaled, y_train)

pred_knn_after = knn_after.predict(X_test_scaled)

acc_knn_after = accuracy_score(
    y_test, pred_knn_after
)

# Decision Tree

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)

acc_dt = accuracy_score(y_test, pred_dt)

# Random Forest

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, pred_rf)

# Classification Summary Table

results = pd.DataFrame({
    "Model": [
        "Logistic Regression (Before Scaling)",
        "Logistic Regression (After Scaling)",
        "KNN (Before Scaling)",
        "KNN (After Scaling)",
        "Decision Tree",
        "Random Forest"
    ],

    "Accuracy": [
        acc_log_before,
        acc_log_after,
        acc_knn_before,
        acc_knn_after,
        acc_dt,
        acc_rf
    ]
})

results["Accuracy (%)"] = (
    results["Accuracy"] * 100
).round(2)

print("\nClassification Model Comparison:")
print(results[["Model", "Accuracy (%)"]])

# Classification report
print("\nLogistic Regression Classification Report:")
print(
    classification_report(
        y_test,
        pred_log_after,
        target_names=wine.target_names
    )
)

# TASK 3: UNSUPERVISED LEARNING
# K-Means Clustering

# Remove target column
X = df.drop(columns=["target"])

# Scale features
scaler_cluster = StandardScaler()

X_scaled = scaler_cluster.fit_transform(X)

# K-Means with 3 clusters
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels
df["cluster"] = clusters

print("\nCluster Labels:")
print(df["cluster"].value_counts())

# Compare clusters with actual wine classes

print("\nCluster vs Actual Wine Class:")

comparison = pd.crosstab(
    df["cluster"],
    df["target"],
    rownames=["K-Means Cluster"],
    colnames=["Actual Wine Class"]
)

print(comparison)

# TASK 4: PCA
# Reduce 13 features to 2 principal components

# Use scaled features
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["target"] = df["target"]

print("\nPCA Dataset:")
print(pca_df.head())

print(
    "\nPC1 Explained Variance:",
    round(pca.explained_variance_ratio_[0] * 100, 2),
    "%"
)

print(
    "PC2 Explained Variance:",
    round(pca.explained_variance_ratio_[1] * 100, 2),
    "%"
)

print(
    "Total Explained Variance:",
    round(pca.explained_variance_ratio_.sum() * 100, 2),
    "%"
)

print("\nProject Completed Successfully!")


Dataset Shape: (178, 14)

First 5 Rows:
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_dilu

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Classification Model Comparison:
                                  Model  Accuracy (%)
0  Logistic Regression (Before Scaling)         97.22
1   Logistic Regression (After Scaling)         97.22
2                  KNN (Before Scaling)         80.56
3                   KNN (After Scaling)         97.22
4                         Decision Tree         94.44
5                         Random Forest        100.00

Logistic Regression Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36


Cluster Labels:
cluster
0    65
2    62
1    51
Name: count, dtype: int64

Cluster vs Actual Wine Class:
Actual Wine Class   0   1   2
K-Means Cluster            